In [ ]:
import time
import numpy as np
import pandas as pd
from collections import Counter
from IPython.display import display
import sklearn
from sklearn import set_config
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import CategoricalNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    KBinsDiscretizer,
    FunctionTransformer
)
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report
)



In [ ]:
# df = pd.read_parquet('data_processed.parquet')

# pipelines

In [ ]:
def analyze_features_for_groups(df, low_card_threshold=20):
    """
    Analyzes columns to map them directly to the Feature Selector Block groups (A-F).
    """
    report = []
    
    # Initialize lists for the copy-paste section
    groups = {
        'A_Numeric': [],
        'B_Binary_Ready': [],
        'C_Low_Card': [],
        'D_High_Card': []
    }
    
    for col in df.columns:
        dtype = df[col].dtype
        suggestion = "Unknown"
        
        # 1. Handle Unhashable / Dictionary types safely
        try:
            num_unique = df[col].nunique()
            unique_vals = df[col].dropna().unique()
        except TypeError:
            num_unique = df[col].astype(str).nunique()
            unique_vals = df[col].astype(str).dropna().unique()
            dtype = 'object (mixed)'

        # 2. Logic to classify into Groups
        is_numeric = pd.api.types.is_numeric_dtype(dtype)
        
        # --- Group B: Binary Ready ---
        # Check if values are strictly {0, 1} or {True, False}
        if num_unique == 2 and set(unique_vals).issubset({0, 1, 0.0, 1.0, False, True}):
            suggestion = "Group B (Binary Ready)"
            groups['B_Binary_Ready'].append(col)
            
        # --- Group A: Continuous Numeric ---
        # If numeric and not binary, and has many unique values -> Continuous
        elif is_numeric and num_unique > low_card_threshold:
            suggestion = "Group A (Numeric)"
            groups['A_Numeric'].append(col)
            
        # --- Group C: Low Cardinality ---
        # Few unique values (Categorical or Discrete Numeric) -> One-Hot candidate
        elif num_unique <= low_card_threshold:
            suggestion = "Group C (Low Cardinality)"
            groups['C_Low_Card'].append(col)
            
        # --- Group D: High Cardinality ---
        # Many unique values (Categorical) -> Ordinal candidate
        else: 
            suggestion = "Group D (High Cardinality)"
            groups['D_High_Card'].append(col)

        # Get examples for display
        examples = list(unique_vals[:3]) if len(unique_vals) > 0 else []

        report.append({
            'Column': col,
            'Unique Count': num_unique,
            'Dtype': dtype,
            'Assigned Group': suggestion,
            'Examples': examples
        })
    
    metrics_df = pd.DataFrame(report).sort_values(by=['Assigned Group', 'Unique Count'])
    
    # --- Print Copy-Paste Blocks ---
    print("="*60)
    print("       🚀 FEATURE SELECTOR COPY-PASTE HELPER")
    print("="*60)
    
    print(f"\n# --- Group A: Continuous Numeric Features (Scale/Passthrough) ---")
    print(f"NUMERIC_FEATS = {groups['A_Numeric']}")
    
    print(f"\n# --- Group B: Binary Features (Already 0/1) ---")
    print(f"BINARY_FEATS = {groups['B_Binary_Ready']}")
    
    print(f"\n# --- Group C: Low Cardinality (For OHE/Tree) ---")
    print(f"CAT_FEATS_LOW_CARD = {groups['C_Low_Card']}")
    
    print(f"\n# --- Group D: High Cardinality (For Ordinal/Native) ---")
    print(f"CAT_FEATS_HIGH_CARD = {groups['D_High_Card']}")
    
    print(f"\n# --- Group E: Aggressively Reduced Categorical ---")
    print(f"# NOTE: You must manually select the 'broad' versions of columns from Groups C/D above.")
    print(f"CAT_FEATS_REDUCED = [] # Fill manually based on your domain logic")

    print(f"\n# --- Group F: Discretized Numeric (For Naive Bayes) ---")
    print(f"# NOTE: Usually identical to Group A (Numeric). Verify logic.")
    print(f"NUMERIC_TO_BIN_FOR_NB = {groups['A_Numeric']}")
    
    print("\n" + "="*60)
    return metrics_df

# Run mapping
group_report = analyze_features_for_groups(df, low_card_threshold=20)
display(group_report)

       🚀 FEATURE SELECTOR COPY-PASTE HELPER

# --- Group A: Continuous Numeric Features (Scale/Passthrough) ---
NUMERIC_FEATS = ['distance', 'latitude', 'longitude', 'hour']

# --- Group B: Binary Features (Already 0/1) ---
BINARY_FEATS = ['driver_substance_abuse', 'is_bidirectional', 'is_weekend', 'is_holiday', 'road_grade_downhill', 'road_grade_hillcrest', 'road_grade_level', 'road_grade_on_bridge', 'road_grade_other', 'road_grade_sag', 'road_grade_uphill', 'road_alignment_curve_left', 'road_alignment_curve_right', 'road_alignment_other', 'road_alignment_straight', 'is_two_way']

# --- Group C: Low Cardinality (For OHE/Tree) ---
CAT_FEATS_LOW_CARD = ['agency_name', 'acrs_report_type', 'hit_run', 'route_type', 'direction', 'distance_unit', 'road_grade', 'at_fault', 'weather', 'surface_condition', 'light', 'junction', 'road_alignment', 'road_condition', 'intersection_type', 'off_road_description', 'municipality', 'surface_condition_simple', 'surface_condition_aggressive', 'crash_year',

,Column,Unique Count,Dtype,Assigned Group,Examples
43,hour,24,int32,Group A (Numeric),"[3, 17, 5]"
11,distance,5868,float64,Group A (Numeric),"[15.55, 0.0, 11.16]"
29,latitude,104390,float64,Group A (Numeric),"[39.20478445, 39.1316597, 39.19877162]"
30,longitude,106561,float64,Group A (Numeric),"[-77.24593535, -77.22564224, -77.15605899]"
22,driver_substance_abuse,2,float64,Group B (Binary Ready),"[1.0, 0.0]"
...,...,...,...,...,...
15,cross_street_name,7511,object,Group D (High Cardinality),"[RIDGE RD (SB/L), PURCHASE ST, GREAT SENECA HWY]"
4,crash_date_time,114133,datetime64[ns],Group D (High Cardinality),"[2025-12-24 03:45:00, 2025-12-22 17:40:00, 202..."
31,geolocation,116230,object (mixed),Group D (High Cardinality),"[{'human_address': '{""address"": """", ""city"": """"..."
1,local_case_number,116903,object,Group D (High Cardinality),"[250057434, 250057180, 250002678]"


In [ ]:
# ==========================================
# 1. CONFIGURATION & FEATURE GROUPS
# ==========================================

# The Target Variable
# Ensure this is label-encoded (0, 1, 2...)
TARGET_COL = 'acrs_report_type'

# --- Group A: Continuous Numeric Features ---
# Use for: Logistic Regression (must be Scaled), RF, LightGBM, CatBoost.
# NOT for: CategoricalNB (it requires Group F instead).
NUMERIC_FEATS = [
    'latitude',
    'longitude',
    'speed_limit',
    # Add other continuous vars like 'distance_from_intersection' if relevant
]

# --- Group B: Binary Features (Already Processed) ---
# Use for: ALL 5 Models.
# These must be strictly 0/1 or True/False integers.
# Do not include features that still need One-Hot Encoding.
BINARY_FEATS = [
    'hit_run',               # Assuming you mapped Yes/No to 1/0
    'is_weekend',            # Created from date
    # 'road_grade_hillcrest' # Only if you decided to keep specific OHE columns
]

# --- Group C: Low Cardinality Categorical (Detailed) ---
# Use for: RF, CatBoost, LightGBM, CategoricalNB.
# Definition: Categories with < 20 unique values.
# Pipeline Action: OHE for RF/LogReg; Ordinal/Integer for CatNB/LGBM.
CAT_FEATS_LOW_CARD = [
    'weather_detailed',      # Full weather list
    'surface_condition',
    'light',
    'traffic_control'
]

# --- Group D: High Cardinality Categorical (Specific) ---
# Use for: CatBoost, LightGBM, Random Forest.
# Definition: Categories with many values (Neighborhoods, Streets).
# Pipeline Action: Ordinal/Target Encoding for RF; Native handling for CatBoost/LGBM.
# Warning: Do NOT use for Logistic Regression (too many dimensions).
CAT_FEATS_HIGH_CARD = [
    'neighborhood_specific',
    'vehicle_make',
    'off_road_description'
]

# --- Group E: Aggressively Reduced Categorical ---
# Use for: Logistic Regression (Critical to avoid overfitting).
# Definition: Broad groupings of the high-cardinality features.
# Pipeline Action: One-Hot Encoding.
CAT_FEATS_REDUCED = [
    'weather_broad',         # e.g., "Clear" vs "Not Clear"
    'neighborhood_zone',     # e.g., "North", "South", "Center"
    'vehicle_type_agg'       # e.g., "Car", "Truck", "Cycle"
]

# --- Group F: Discretized Numeric (Binned) ---
# Use for: CategoricalNB ONLY.
# Definition: Numeric features converted to categorical bins (e.g., Speed 0-30 -> 'Low').
# Note: You need to create these columns in preprocessing OR use a KBinsDiscretizer in pipeline.
# List here the *original* numeric names you want to bin inside the pipeline:
NUMERIC_TO_BIN_FOR_NB = [
    'speed_limit',
    'latitude',   # Binning coordinates creates "Grid Cells"
    'longitude'
]

# --- Sanity Check ---
print(f"Target: {TARGET_COL}")
print(f"Group A (Numeric): {len(NUMERIC_FEATS)} features")
print(f"Group B (Binary): {len(BINARY_FEATS)} features")
print(f"Group C (Low Card): {len(CAT_FEATS_LOW_CARD)} features")
print(f"Group D (High Card): {len(CAT_FEATS_HIGH_CARD)} features")
print(f"Group E (Reduced): {len(CAT_FEATS_REDUCED)} features")
print(f"Group F (To Bin): {len(NUMERIC_TO_BIN_FOR_NB)} features")

In [ ]:
set_config(transform_output="pandas") # For easier debugging with DataFrames and because we work with catecory data type

# ==============================
# CATBOOST & LIGHTGBM PIPELINES 
# ===============================

native_preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', NUMERIC_FEATS),
        ('cat', 'passthrough', CAT_FEATS_LOW_CARD + CAT_FEATS_HIGH_CARD),
        ('bin', SimpleImputer(strategy='most_frequent'), BINARY_FEATS)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

# --- CatBoost ---
cb_pipeline = Pipeline([
    ('preprocessor', native_preprocessor),
    ('model', CatBoostClassifier(
        auto_class_weights='Balanced', # Increases penalty for mistakes on rare classes
        verbose=50,
        allow_writing_files=False,
        loss_function='MultiClass', # penalizes the model for being "confident and wrong" + softmax
        random_state=42
    ))
])

# --- LightGBM ---
lgbm_pipeline = Pipeline([
    ('preprocessor', native_preprocessor),
    ('model', LGBMClassifier(
        class_weight='balanced', # Increases penalty for mistakes on rare classes
        verbose=10,
        n_jobs=-1,
        objective='multiclass', # multi-class classification, softmax
        num_class=3, 
        metric='multi_logloss', # penalizes the model for being "confident and wrong"
        random_state=42
    ))
])

In [ ]:
# ======================================================
# PREPROCESSOR FOR RANDOM FOREST
# ======================================================
rf_preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), NUMERIC_FEATS), #why median? # no scaling needed for RF
        ('bin', SimpleImputer(strategy='most_frequent'), BINARY_FEATS),

        ('cat_low', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
        ]), CAT_FEATS_LOW_CARD),

        ('cat_high', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))  # We compromise on Ordinal (integers) to keep the model compact.
        ]), CAT_FEATS_HIGH_CARD)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

# ======================================================
# MODEL PIPELINE: RANDOM FOREST
# ======================================================
rf_pipeline = Pipeline([
    ('preprocessor', rf_preprocessor),
    ('model', RandomForestClassifier(
        class_weight='balanced',  # importent for rare classes

        n_estimators=200,      
        max_depth=15,          # Limit depth to prevent Overfitting on noisy data
        min_samples_split=10,  # Require at least 10 samples to split a node

        n_jobs=-1,         
        random_state=42,
        verbose=0           
    ))
])

In [ ]:
# ======================================================
# PREPROCESSOR FOR LOGISTIC REGRESSION
# ======================================================
lr_preprocessor = ColumnTransformer(
    transformers=[
        # Numeric: scale + impute
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()) 
        ]), NUMERIC_FEATS),

        ('bin', SimpleImputer(strategy='most_frequent'), BINARY_FEATS),

        # Low Cardinality Categorical - OHE
        ('cat_low', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), CAT_FEATS_LOW_CARD),

        # high cardinaliry - Reduced Categorical features
        ('cat_reduced', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), CAT_FEATS_REDUCED) 
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

# ======================================================
# MODEL PIPELINE: LOGISTIC REGRESSION
# ======================================================
lr_pipeline = Pipeline([
    ('preprocessor', lr_preprocessor),
    ('model', LogisticRegression(
        # --- Solver & Convergence ---
        solver='lbfgs',          # The standard engine (Stable, handles Multiclass, supports L2, suitable for our data size)
        max_iter=500,            # Changed from default (100) to ensure convergence on imbalanced data
        
        # --- Class Imbalance ---
        class_weight='balanced', # Forces the model to give more weight to rare classes, maybe we will manually weight later
        
        # --- Technical ---
        n_jobs=-1,             
        random_state=42          
        
        # --- Hidden Defaults ---
        # penalty='l2' (Standard Ridge Regularization is on)
        # C=1.0 (Moderate regularization strength)
        # multi_class='auto' (Automatically selects Softmax/Multinomial)
    ))
])

In [ ]:
def shift_unknown_to_zero(X):
    '''Shifts all values by +1 to ensure non-negative integers.

    This maps OrdinalEncoder's unknown value (-1) to 0, preventing 
    CategoricalNB from crashing on negative inputs.'''
    return X + 1

# ======================================================
# PREPROCESSOR FOR NAIVE BAYES - OPTIMIZED
# ======================================================
nb_preprocessor = ColumnTransformer(
    transformers=[
        # --- Coordinates: 20 bins for high spatial resolution ---
        ('coords_binned', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('discretizer', KBinsDiscretizer(
                n_bins=20,
                encode='ordinal', 
                strategy='quantile'
            ))
        ]), ['latitude', 'longitude']),
        
        # --- Other Numeric Features: 10 bins ---
        ('other_numeric_binned', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('discretizer', KBinsDiscretizer(
                n_bins=10,
                encode='ordinal', 
                strategy='quantile'
            ))
        ]), [col for col in NUMERIC_TO_BIN_FOR_NB if col not in ['latitude', 'longitude']]),
        
        # --- Binary Features ---
        ('bin', SimpleImputer(strategy='most_frequent'), BINARY_FEATS),
        
        # --- Categorical (Low + High Cardinality) ---
        ('cat_all', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('ordinal', OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1  # NB cant handle negative values, so we will shift later
            )),
            ('shift', FunctionTransformer(
                shift_unknown_to_zero,  # unknowns -> 0 , all others +1
                validate=False
            ))
        ]), CAT_FEATS_LOW_CARD + CAT_FEATS_HIGH_CARD)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

# ======================================================
# MODEL PIPELINE: CATEGORICAL NAIVE BAYES
# ======================================================
nb_pipeline = ImbPipeline([
    ('preprocessor', nb_preprocessor),
    ('resampler', RandomOverSampler(
        sampling_strategy='not majority', # maybe we will try dictionary later
        random_state=42
    )),
    
    ('model', CategoricalNB(
        alpha=1.0,  # Laplace smoothing to avoid zero probabilities. high -> more wheigt to prior. low -> more weight to data.
    ))
])


## הרצה והשוואה

In [ ]:
# ======================================================
# 1. DATA SPLIT (Stratified)
# ======================================================
# החלוקה מתבצעת עם stratify כדי לשמור על יחס המחלקות בטסט
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ======================================================
# 2. AUTOMATIC MINORITY CLASS IDENTIFICATION
# ======================================================
class_counts = Counter(y_train)
minority_class = min(class_counts, key=class_counts.get)
all_classes = sorted(list(class_counts.keys()))

print(f"Dataset Stats:")
for cls, count in sorted(class_counts.items()):
    print(f"  Class {cls}: {count:,} ({count/len(y_train):.2%})")
print(f"Target Minority Class: {minority_class}\n")

# ======================================================
# 3. BENCHMARK FUNCTION
# ======================================================
def run_benchmark(pipelines_dict, X_train, y_train, X_test, y_test, minority_class):
    """
    Evaluates multiple pipelines and returns a performance leaderboard.
    Focuses on Macro-averaging and minority class performance.
    """
    results = []
    
    for name, pipe in pipelines_dict.items():
        print(f"Processing: {name}...")
        
        # Training
        start_time = time.time()
        pipe.fit(X_train, y_train)
        train_time = time.time() - start_time
        
        # Prediction
        y_pred = pipe.predict(X_test)
        
        # Global Metrics
        f1_macro = f1_score(y_test, y_pred, average='macro')
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        
        # Per-class Metrics
        # labels=pipe.classes_ ensures indices match the model's internal mapping
        prec, rec, f1_per_class, _ = precision_recall_fscore_support(
            y_test, y_pred, labels=pipe.classes_, zero_division=0
        )
        
        # Identify index of minority class in the results arrays
        min_idx = list(pipe.classes_).index(minority_class)
        
        # Store results
        results.append({
            'Model': name,
            'F1 Macro': round(f1_macro, 4),
            'Balanced Acc': round(bal_acc, 4),
            'Min. Recall': round(rec[min_idx], 4),
            'Min. Precision': round(prec[min_idx], 4),
            'Min. F1': round(f1_per_class[min_idx], 4),
            'Time (s)': round(train_time, 2)
        })
        
        # Intermediate output for tracking
        print(f"Classification Report for {name}:")
        print(classification_report(y_test, y_pred, zero_division=0, digits=3))
        print("-" * 50)

    # Leaderboard creation
    leaderboard = pd.DataFrame(results).sort_values('F1 Macro', ascending=False)
    return leaderboard

# ======================================================
# 4. EXECUTION
# ======================================================
pipelines = {
    'LR': lr_pipeline,
    'NB': nb_pipeline,
    'RF': rf_pipeline,
    'CB': cb_pipeline,  
    'LGBM': lgbm_pipeline 
}

leaderboard = run_benchmark(
    pipelines, 
    X_train, y_train, 
    X_test, y_test, 
    minority_class
)

print("\nFINAL LEADERBOARD")
print("=" * 70)
print(leaderboard.to_string(index=False))